# Stochastic Calculus, Step by Step · Lesson 11: Martingales and risk-neutral pricing

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Martingales, submartingales, supermartingales · 3. Discounted prices are martingales under p̃ · 4. The pricing formula · 5. When the risk-neutral probability is not unique · 6. Markov or path-dependent · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 11,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. Martingales, submartingales, supermartingales

An adapted process $M_n$ is a **martingale** if the best forecast of tomorrow is today: $E[M_{n+1} \mid \mathcal F_n] = M_n$. A **submartingale** tends to rise ($\ge$), a **supermartingale** tends to fall ($\le$). On the tree, $E[X_{n+1} \mid \mathcal F_n] = g\,X_n$ for the processes below, where $g$ is the one-step growth factor of the forecast, so the type is read off from $g$:

In [ ]:
pt = (R - d) / (u - d)
kind = lambda g: "martingale" if abs(g - 1) < 1e-12 else ("submartingale" if g > 1 else "supermartingale")
cases = [("S_n", "p = 0.6", p * u + (1 - p) * d),
         ("S_n", "p = 0.5", 0.5 * u + 0.5 * d),
         ("S_n", "p̃", pt * u + (1 - pt) * d),
         ("S_n / R^n", "p = 0.6", (p * u + (1 - p) * d) / R),
         ("S_n / R^n", "p̃", (pt * u + (1 - pt) * d) / R),
         ("R^n / S_n", "p̃", R * (pt / u + (1 - pt) / d))]
out["kinds"] = [{"process": a, "measure": b_, "g": g, "kind": kind(g)} for a, b_, g in cases]
out["mart"] = {"pt": pt, "g_p": p * u + (1 - p) * d, "g_p_disc": (p * u + (1 - p) * d) / R, "R": R}
pd.DataFrame(out["kinds"])

## 3. Discounted prices are martingales under p̃

Under $\tilde p$ the share grows on average exactly like cash, so the **discounted** share price $S_n/R^n$ is a martingale. So is the discounted value of any option priced by backward induction: each step of the induction *is* the martingale property. Check it at every node for the three-step call:

In [ ]:
S = lambda n, k_: S0_ * u**k_ * d ** (n - k_)
V = {(steps, k_): max(S(steps, k_) - K, 0.0) for k_ in range(steps + 1)}
for n in range(steps - 1, -1, -1):
    for k_ in range(n + 1):
        V[(n, k_)] = (pt * V[(n + 1, k_ + 1)] + (1 - pt) * V[(n + 1, k_)]) / R
gaps = []
for n in range(steps):
    for k_ in range(n + 1):
        today = V[(n, k_)] / R**n
        forecast = (pt * V[(n + 1, k_ + 1)] + (1 - pt) * V[(n + 1, k_)]) / R ** (n + 1)
        gaps.append(abs(forecast - today))
out["disc"] = {"max_gap": max(gaps), "V0": V[(0, 0)],
               "nodes": [{"t": n, "k": k_, "price": S(n, k_), "value": V[(n, k_)], "disc": V[(n, k_)] / R**n}
                         for n in range(steps + 1) for k_ in range(n + 1)]}
out["disc"]["max_gap"]

## 4. The pricing formula

Since $V_n/R^n$ is a martingale under $\tilde p$ and ends at the payoff, $V_n = \tilde E[V_N \mid \mathcal F_n]/R^{N - n}$, and today $V_0 = \tilde E[V_N]/R^N$. Computed world by world, as an average over the eight paths:

In [ ]:
paths = [{"world": w, "q": prob(w, pt), "P": prob(w), "S3": price(w, 3), "payoff": max(price(w, 3) - K, 0.0)} for w in worlds]
v_paths = sum(x["q"] * x["payoff"] for x in paths) / R**steps
real = sum(x["P"] * x["payoff"] for x in paths) / R**steps
assert abs(v_paths - V[(0, 0)]) < 1e-12
out["formula"] = {"paths": paths, "V0": v_paths, "real_disc": real, "ES3_rn": sum(x["q"] * x["S3"] for x in paths),
                  "S0_R3": S0_ * R**steps}
pd.DataFrame(paths).round(4)

## 5. When the risk-neutral probability is not unique

The **fundamental theorems of asset pricing**: (1) a market has no arbitrage exactly when a risk-neutral probability exists; (2) every payoff can be copied (the market is **complete**) exactly when it is unique. Give the share a third outcome, staying at 100, and the risk-neutral conditions $q_u + q_m + q_d = 1$, $1.1q_u + q_m + 0.9q_d = R$ leave one free parameter. Each choice gives a different call price, all free of arbitrage: no copy exists, so no single price is forced.

In [ ]:
lo = (R - 1) / (u - 1)          # the two conditions give q_u - q_d = lo; q_d > 0 needs q_u > lo
hi = (1 + lo) / 2               # q_m = 1 - 2 q_u + lo > 0 needs q_u < hi
qs = np.linspace(lo, hi, 11)[1:-1]            # the end points give a zero probability
tri_rows = [{"q_u": q, "q_m": 1 - 2 * q + lo, "q_d": q - lo, "call": q * (S0_ * u - K) / R} for q in qs]
assert all(abs(x["q_u"] * u + x["q_m"] + x["q_d"] * d - R) < 1e-12 for x in tri_rows)
out["incomplete"] = {"q_lo": lo, "q_hi": hi, "call_lo": lo * (S0_ * u - K) / R, "call_hi": hi * (S0_ * u - K) / R,
                     "rows": tri_rows}
pd.DataFrame(tri_rows).round(4)

## 6. Markov or path-dependent

The share price is **Markov**: its future depends on the past only through today's price, which is why the tree can merge HT and TH into one node. A payoff that depends on the whole path breaks this. An **Asian call** pays $\max(A - 105, 0)$ on the average price $A = (S_1 + S_2 + S_3)/3$, and HTH and THH end at the same node with different averages. Such a payoff is priced over all $2^N$ paths, or on an enlarged state (price and running total) that is Markov again.

In [ ]:
asian, look = [], []
for w in worlds:
    s = [price(w, n) for n in range(steps + 1)]
    A = sum(s[1:]) / steps
    asian.append({"world": w, "q": prob(w, pt), "path": [round(x, 2) for x in s], "A": A, "payoff": max(A - K, 0.0),
                  "M": max(s), "look": max(max(s) - K, 0.0)})
asian_price = sum(x["q"] * x["payoff"] for x in asian) / R**steps
look_price = sum(x["q"] * x["look"] for x in asian) / R**steps
counts = [{"N": n, "nodes": n + 1, "tree_nodes": (n + 1) * (n + 2) // 2, "paths": 2**n} for n in (3, 10, 20, 50)]
out["path"] = {"rows": asian, "asian": asian_price, "lookback": look_price, "counts": counts,
               "A_HTH": next(x["A"] for x in asian if x["world"] == "HTH"),
               "A_THH": next(x["A"] for x in asian if x["world"] == "THH")}
pd.DataFrame(asian).round(4)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** Under the real probability $p = 0.6$, show that $S_n$ is a submartingale. For which $p$ is $S_n$ a martingale? Under $\tilde p$, is $S_n$ itself (not discounted) a martingale?

**Exercise 2.** Prove that $S_n/R^n$ is a martingale under $\tilde p$ for any one-step factors with $d < R < u$, using *taking out what is known*.

**Exercise 3.** Show that the backward-induction values satisfy $\tilde E[V_{n+1}/R^{n+1} \mid \mathcal F_n] = V_n/R^n$, and deduce $V_0 = \tilde E[V_3]/R^3$ with the tower rule.

**Exercise 4.** Show that a martingale has constant expectation, $E[M_n] = M_0$, and that $E[M_m \mid \mathcal F_n] = M_n$ for every $m > n$, not just $m = n + 1$.

**Exercise 5.** A lookback call pays $\max(M_3 - 105, 0)$, where $M_3$ is the highest of $S_0, \dots, S_3$. Price it by hand over the eight paths under $\tilde p$.

**Exercise 6.** Let $X_n$ be the value of a self-financing portfolio of shares and cash. Show that $X_n/R^n$ is a martingale under $\tilde p$, and use it to show that no such portfolio can start at 0, never end below 0, and end above 0 with positive probability.

**Exercise 7.** (Code) Price the Asian call ($K = 105$, average of $S_1, \dots, S_N$) on a CRR tree with the course's numbers by brute force over all $2^N$ paths, for $N = 4, 8, 12, 16$. How does the number of paths grow, and how many paths would $N = 252$ (daily steps) need?

**Exercise 8.** (Code) On a 50-step CRR tree for the course's call, check the martingale property of the discounted value at every node, and print the largest gap.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")